# Import

In [1]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py

%load_ext autoreload
%autoreload 2


%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)




# Standard plotting
def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=None,savefig=False,format='png',
                                export=True)

# Load Sinq

In [2]:
# sinq = Sinq(sinqname='Figure3')
# print(sinq.__repr__())

# sinq.df
# sinq_g = subset_sinq(source=sinq_g,rows=g_idx,name='Figure6_mutant_pca')

sinq = Sinq(sinqname='dataset_3_genetic_manipulations')
print(sinq.__repr__())
# sinq.display_df()

Found data directory: D:\Data
Sinq(dataset_3_genetic_manipulations, 37 rows x 33 params): 0 empties; ['241205_F2_C1', '241205_F3_C1', '241219_F1_C1', '241219_F2_C1', '241220_F1_C1', '241220_F2_C1', '250127_F1_C1', '250127_F2_C1', '250128_F1_C1', '250204_F1_C1', '250204_F2_C1', '250211_F2_C1', '250211_F3_C1', '250317_F1_C1', '250317_F2_C1', '250318_F1_C1', '250318_F2_C1', '250318_F3_C1', '250320_F1_C1', '250320_F2_C1', '250325_F1_C1', '250423_F1_C1', '250423_F2_C1', '250506_F1_C1', '250506_F2_C1', '250507_F1_C1', '250507_F2_C1', '250509_F2_C1', '250514_F1_C1', '250515_F1_C1', '251114_F1_C2', '251217_F2_C1', '260527_F1_C1', '260527_F4_C1', '260605_F1_C1', '260605_F2_C1', '260605_F3_C1'] x ['Table', 'parquet', 'genotype', 'notes', 'duration', 'rms_velocity', 'outcome_fractions_no_as_no_mv', 'outcome_fractions_as_off', 'outcome_fractions_rest', 'outcome_fractions_no_as_mv', 'outcome_fractions_probe', 'outcome_fractions_as_off_late', 'outcome_fractions_timeout_fail', 'outcome_fractions_time

In [3]:
# sinq.restore_table(sinq.df.index[0])
# sinq.add_column('rest_time_fraction')
# sinq.add_column('drift_time_fraction')
# sinq.add_column('move_time_fraction')

# sinq.restore_table('250320_F2_C1')
# sinq.sync_row('250320_F1_C1', overwrite=False)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\250320\\250320_F2_C1\\LEDFlashTriggerPiezoControl_250320_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=238, 250320_F2_C1, dT=16.49746, ex=True)]
Getting all meta keys
['as_outcome', 'fiberLED', 'filtercube_status']
Found 2 target positions - Counter({('hi', 640.0, 360.0, -280.0, 60.0): 506, ('lo', 640.0, 460.0, -180.0, 60.0): 489})
Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\250320\\250320_F1_C1\\LEDFlashTriggerPiezoControl_250320_F1_C1_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=147, 250320_F1_C1, dT=707.19386, ex=True), Trial(trial=432, 250320_F1_C1, dT=8.86788, ex=True), Trial(trial=433, 250320_F1_C1, dT=9.7624, ex=True), Trial(trial=434, 250320_F1_C1, dT=10.29686, ex=True), Trial(trial=435, 250320_F1_C1, dT=9.06674, ex=True), Trial(trial=436, 250320_F1_C1, dT=10.24008, ex=True), Trial(trial=437, 250320_F1_C1, dT=8.70014, ex=True), Trial(trial=438, 250320_F1_C1, dT=8

Table                             Day 250320, F1, C1: LEDFlashTriggerPiezoContro...
parquet                           LEDFlashTriggerPiezoControl_250320_F1_C1_Table...
genotype                                                       +;iav-GAL4_UAS-Kir21
notes                             {'text': 'IAV Kir2.1 in [+] background', 'tags...
duration                                                                 7482.59076
rms_velocity                                                          249119.133081
outcome_fractions_no_as_no_mv                                              0.349229
outcome_fractions_as_off                                                   0.274895
outcome_fractions_rest                                                     0.106592
outcome_fractions_no_as_mv                                                 0.119215
outcome_fractions_probe                                                     0.01683
outcome_fractions_as_off_late                                              0

# Behavior with drivers (WT or w)

## Green to Blue LED behavior

### 250127_F1_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:
id = '250127_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_250127_F1_C1_Table.parquet')
# T.assign_column_value('filtercube_status','green',trial_min=4,trial_max=416, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','blue',trial_min=417, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.plot_outcomes(savefig=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')

# df_filter = {'pyasState':'hi','_filtercube_status_cat':'green'}
# early_index = T.df.index[0:100]
# late_index = T.df.index[100:]

# T.plot_probe_distribution(filter=None,index=None,bin_min=-500,bin_max=10,savefig=True)
# plt.close()

# plt.close()

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','green','blue'],
)
# sinq.add_table(table=T)
# sinq.df

### 250127_F2_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:

id = '250127_F2_C1'
# T = Table('LEDFlashTriggerPiezoControl_250127_F2_C1_Table.parquet')
# T.assign_column_value('filtercube_status','green',trial_min=4,trial_max=415, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','blue',trial_min=417, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','green','blue'],
)

# sinq.add_table(table=T)
# sinq.df

### 250128_F1_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:
id = '250128_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_250128_F1_C1_Table.parquet')
# T.assign_column_value('filtercube_status','green',trial_min=2,trial_max=582, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','blue',trial_min=583, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','green','blue'],
)

# T.exclude_list_of_trials(poor_tracking.index,'poor_tracking')
# solution_ran_out = T.df.loc[350:374]
# solution_ran_out.index
# T.exclude_list_of_trials(solution_ran_out.index,'solution_ran_out')
# sinq.add_table(table=T)
# sinq.df

### 250204_F1_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:
id = '250204_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_250204_F1_C1_Table.parquet')
# T.add_df_category('green',trial_min=1,trial_max=576,categories='_filtercube_status_cat')
# T.add_df_category('blue',trial_min=577,categories='_filtercube_status_cat')
# T.assign_column_value('filtercube_status','green',trial_min=1,trial_max=576, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','blue',trial_min=577, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','green','blue'],
)

### 250204_F2_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:
id = '250204_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_250204_F2_C1_Table.parquet')
# T.add_df_category('green',trial_min=1,trial_max=408,categories='_filtercube_status_cat')
# T.add_df_category('blue',trial_min=409,categories='_filtercube_status_cat')
# T.assign_column_value('filtercube_status','green',trial_min=1,trial_max=408, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','blue',trial_min=409, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','green','blue'],
)

## Blue to Green 

### 250211_F2_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:
id = '250211_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_250211_F2_C1_Table.parquet')
# T.add_df_category('blue',trial_min=1,trial_max=354,categories='_filtercube_status_cat')
# T.add_df_category('green',trial_min=355,categories='_filtercube_status_cat')
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=354, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','green',trial_min=355, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','blue','green',],
)

### 250211_F3_C1 - w[1118];pJFRC7;31H05-Gal4

In [ ]:
id = '250211_F3_C1'

# T = Table('LEDFlashTriggerPiezoControl_250211_F3_C1_Table.parquet')
# # T.add_df_category('blue',trial_min=1,trial_max=489,categories='_filtercube_status_cat')
# # T.add_df_category('green',trial_min=490,categories='_filtercube_status_cat')
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=489, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','green',trial_min=490, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'w',
    tags=['w','blue','green',],
)

# IAV Kir2.1 in [w] background

These are with w;;UAS-Kir2.1

## 241205_F2_C1 - w;+;iav-gal4/uas-kir2.1

In [ ]:
id = '241205_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_241205_F2_C1_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# # T.assign_column_value('filtercube_status','green',trial_min=490, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [w] background',
    tags=['w','iav','blue','green',],
)

## 241205_F3_C1 - w;+;iav-gal4/uas-kir2.1

In [ ]:
id = '241205_F3_C1'

# T = Table('LEDFlashTriggerPiezoControl_241205_F3_C1_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# # T.assign_column_value('filtercube_status','green',trial_min=490, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_compute(sinq,id)
# run_plots(T)

# T.compute_trial_method(method_name='probe_positive_effort')
# T.plot_trial_computations(method_name='probe_positive_effort')

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [w] background',
    tags=['w','iav','blue','green',],
)

## 241219_F1_C1 - w;+;iav-gal4/uas-kir2.1

Blue then Green. 


In [ ]:
id = '241219_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_241219_F1_C1_Table.parquet')

# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=624, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','green',trial_min=625,trial_max=800, write_to_hdf5=True) 
# T.assign_column_value('filtercube_status','blue',trial_min=801, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [w] background',
    tags=['w','iav','blue','green',],
)

## 241219_F2_C1 - w;+;iav-gal4/uas-kir2.1

Blue then Green. 


In [ ]:
id = '241219_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_241219_F2_C1_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=520, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','green',trial_min=521, write_to_hdf5=True) 
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [w] background',
    tags=['w','iav','blue','green',],
)

## 241220_F1_C1 - w;+;iav-gal4/uas-kir2.1

Green then Blue then Green. 


In [ ]:
id = '241220_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_241220_F1_C1_Table.parquet')
# T.assign_column_value('filtercube_status','green',trial_min=1,trial_max=104, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','blue',trial_min=105,trial_max=566, write_to_hdf5=True) 
# T.assign_column_value('filtercube_status','green',trial_min=567, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
# id = '241220_F1_C1'
# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [w] background',
    tags=['w','iav','green','blue',],
)

## 241220_F2_C1 - w;+;iav-gal4/uas-kir2.1

Blue then Green. 


In [ ]:
id = '241220_F2_C1'

# # T = Table('LEDFlashTriggerPiezoControl_241220_F2_C1_Table.parquet')

# T.assign_column_value('filtercube_status','green',trial_min=1,trial_max=104, write_to_hdf5=True)
# T.assign_column_value('filtercube_status','green',trial_min=105,trial_max=572, write_to_hdf5=True) 
# T.assign_column_value('filtercube_status','blue',trial_min=573, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_compute(sinq,id)
# run_plots(T)


sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [w] background',
    tags=['w','iav','green','blue',],
)

# IAV Kir2.1 in [+] background

## Blue

### 250317_F1_C1 - +;+;iav-gal4/uas-kir2.1

Blue


In [ ]:

# T = Table('LEDFlashTriggerPiezoControl_250317_F1_C1_Table.parquet')
# # T.open_notes_files()
# # T.exclude_list_of_trials(T.df.loc[83:86].index,reason='probe slipped, can kick')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
id = '250317_F1_C1'
# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.clear_note(id)
sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','blue',],
)

### 250317_F2_C1 - +;+;iav-gal4/uas-kir2.1

Blue.


In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250317_F2_C1_Table.parquet')
# # T.open_notes_files()
# # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# # fig,ax = T.plot_outcomes(savefig=True,format='png')
# # fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# # sinq.add_table(table=T)
# # sinq.df
id = '250317_F2_C1'
# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','blue',],
)

### 250318_F1_C1 - +;+;iav-gal4/uas-kir2.1

Blue


In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250318_F1_C1_Table.parquet')
# # T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
id = '250318_F1_C1'

# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','blue',],
)

### 250318_F2_C1 - +;+;iav-gal4/uas-kir2.1

Blue


In [ ]:
id = '250318_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_250318_F2_C1_Table.parquet')
# # T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','blue',],
)

### 250318_F3_C1 - +;+;iav-gal4/uas-kir2.1

Blue


In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250318_F3_C1_Table.parquet')
# # T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
id = '250318_F3_C1'

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','blue',],
)

### 250423_F1_C1 - +;+;iav-gal4/uas-kir2.1

Blue


In [ ]:
id = '250423_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_250423_F1_C1_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','blue',],
)

## Green (4)

In [ ]:
# Standard compute
def run_green_trials_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
        # endofdaytrials = T.df.loc[1081:]
        # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
        T.as_outcome_recompute()
        sinq.add_table(table=T)
        sinq.df
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

### 250320_F1_C1 - +;+;iav-gal4/uas-kir2.1

Green


In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250320_F1_C1_Table.parquet')
# # T.open_notes_files()
# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
id = '250320_F1_C1'
# sinq,T = run_green_trials_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','green',],
)

### 250320_F2_C1 - +;+;iav-gal4/uas-kir2.1

Green


In [ ]:
id = '250320_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_250320_F2_C1_Table.parquet')

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_green_trials_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','green',],
)

In [ ]:
sinq.display_df()['tags']

### 250325_F1_C1 - +;+;iav-gal4/uas-kir2.1

In [ ]:
id = '250325_F1_C1'

# T = Table('LEDFlashTriggerPiezoControl_250325_F1_C1_Table.parquet')
# # T.open_notes_files()
# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# sinq,T = run_green_trials_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','green',],
)

### 250423_F2_C1 - +;+;iav-gal4/uas-kir2.1

Green


In [ ]:
id = '250423_F2_C1'

# T = Table('LEDFlashTriggerPiezoControl_250423_F2_C1_Table.parquet')
# # T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df
# id = '250423_F2_C1'
# sinq,T = run_green_trials_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'IAV Kir2.1 in [+] background',
    tags=['+','iav','green',],
)

# IAV > UAS-TeTx

## 251217_F2_C1

In [ ]:
id = '251217_F2_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()

# run_plots(T)

# sinq.add_table(table=T)
# sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'IAV TeTx in [+] background',
    tags=['+','iav','blue','TeTx'],
)

## 251218_F4_C1

In [ ]:
# id = '251218_F4_C1'
# Not adding it. 251217_F2_C1 is better, but I don't think this construct worked here. Deriving the WT white background 
# again, and using Kir2.1 in the mean time. 

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()

# run_plots(T)

# sinq.add_table(table=T)
# sinq.drop_tables()

# sinq.set_note(
#     dayflycell = id,
#     text= 'IAV TeTx in [+] background',
#     tags=['+','iav','green',],
# )

# TH-Gal4>GFP (using WT white allele)

# TH-Gal4>Kir2.1

## Blue

### 250506_F1_C1

In [ ]:
id = '250506_F1_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')

# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()

# sinq.add_table(table=T)

# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'TH>Kir2.1',
    tags=['DANs','TH','PPL','blue'],
)

## Green

### 250506_F2_C1

In [ ]:
id = '250506_F2_C1'

# I guess I'm not adding this one here. it's part of dataset1 because it learns so well. Ignore it.

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
sinq.add_table(table=T)

sinq.clear_note(dayflycell = id)
sinq.set_note(
    dayflycell = id,
    text= 'TH>Kir2.1',
    tags=['DANs','TH','PPL','green'],
)

sinq.display_df(show_tags=True)

# T = sinq.restore_table(dayflycell=id,overwrite=True)
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# run_plots(T)

# sinq.delete_table(id)

### 250507_F1_C1

In [ ]:
id = '250507_F1_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'TH>Kir2.1',
    tags=['DANs','TH','PPL','green'],
)

### 250507_F2_C1

In [ ]:
id = '250507_F2_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'TH>Kir2.1',
    tags=['DANs','TH','PPL','green'],
)

### 250509_F1_C1

In [ ]:
id = '250509_F1_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

# T = sinq.restore_table(id)

# poor_tracking = T.df.loc[310:]
# poor_tracking.index
# T.exclude_list_of_trials(poor_tracking.index,'poor_tracking')
# T.extract_trial_properties()

# sinq.set_note(
#     dayflycell = id,
#     text= 'TH>Kir2.1',
#     tags=['DANs','TH','PPL','green'],
# )

# Don't use this one, pulling too much, maybe interesting, but do this with new flies

# run_plots(T)

sinq.delete_table(id)

In [ ]:
sinq.df.index

### 250509_F2_C1

In [ ]:
id = '250509_F2_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# # T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'TH>Kir2.1',
    tags=['DANs','TH','PPL','green'],
)

### 250514_F1_C1 Training.  Part of the control dataset for some reason?

In [ ]:
id = '250514_F1_C1'

# Note, tried training the fly initially.

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# sinq.add_table(table=T)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= 'TH>Kir2.1',
    tags=['DANs','TH','PPL','green'],
)

# +;TH-Gal4>GFP

### 250515_F1_C1

In [ ]:
id = '250515_F1_C1'

# Note, tried training the fly initially.

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# sinq.add_table(table=T)
# sinq.df
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

# T = sinq.restore_table(id)
# poor_tracking = T.df.loc[750:]
# poor_tracking.index
# T.exclude_list_of_trials(poor_tracking.index,'poor_tracking')
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

sinq.set_note(
    dayflycell = id,
    text= 'TH>GFP',
    tags=['control'],
)

# 58E02 Manipulations

## 58E02>UAS-TeTx - TeTx silencing

### 251114_F1_C2


In [ ]:
id = '251114_F1_C2'

# sinq,T = run_compute(sinq,id)
# run_plots(T)

sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Tetx',
    tags=['DANs','PAM'],
)

### 260605_F1_C1


In [ ]:
# Standard plotting
def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=None,savefig=False,format='png',
                                export=True)

In [ ]:
id = '260605_F1_C1'

# T = T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# run_plots(T)

# sinq.add_table(table=T)
# # sinq.df
sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Tetx',
    tags=['DANs','PAM'],
)

In [ ]:
id = '260605_F2_C1'

# T = T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# run_plots(T)

# sinq.add_table(table=T)
# # sinq.df
sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Tetx',
    tags=['DANs','PAM'],
)

In [ ]:
id = '260605_F3_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# end_of_day_trials = T.df.loc[806:]
# T.exclude_list_of_trials(end_of_day_trials.index,'just not watching')
# T.exclude_trials()

# T.extract_trial_properties()
# run_plots(T)

# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Tetx',
    tags=['DANs','PAM'],
)

## 58E02>UAS-Kir2.1 - Kir2.1 silencing

### 261102_F1_C1
Purpose "Get more PAM flies". No EMG in femur. Pyas stopped at ~374-378 and again ~494 (restarted). probeZero changed to 630 at trial 508.

In [5]:
# [D3-261102_F1_C1]
# NO TABLE YET: build LEDFlashTriggerPiezoControl_261102_F1_C1_Table in MATLAB first, then uncomment.
id = '261102_F1_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# # # Pyas dropped out at 374-378 and ~494: check those trials came out as info / excluded, e.g.
# T.exclude_list_of_trials(T.df.loc[374:378].index, 'pyas dropout'); T.exclude_trials()
# T.extract_trial_properties()
# run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Kir2.1',
    tags=['DANs', 'PAM', '58E02', 'Kir2.1'],
)
# sinq.display_df(show_tags=True)

Adding Table for 261102_F1_C1
Adding parquet for 261102_F1_C1
Adding genotype for 261102_F1_C1
Adding duration for 261102_F1_C1
Adding rms_velocity for 261102_F1_C1
Adding outcome_fractions_no_as_no_mv for 261102_F1_C1
Adding outcome_fractions_as_off for 261102_F1_C1
Adding outcome_fractions_rest for 261102_F1_C1
Adding outcome_fractions_no_as_mv for 261102_F1_C1
Adding outcome_fractions_probe for 261102_F1_C1
Adding outcome_fractions_as_off_late for 261102_F1_C1
Adding outcome_fractions_timeout_fail for 261102_F1_C1
Adding outcome_fractions_timeout for 261102_F1_C1
Adding outcome_fractions_info for 261102_F1_C1
Adding lo_state_median_position for 261102_F1_C1
pyasState
Using existing probe bins
Adding hi_state_median_position for 261102_F1_C1
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
Adding hi_lo_shift for 261102_F1_C1
Adding hi_state_on_target for 261102_F1_C1
pyasState
Using existing probe bins
On target ([-284.] - [-2

### 261102_F2_C1
Purpose "silence PAMs". Pyas was quitting that day.

In [6]:
# [D3-261102_F2_C1]
id = '261102_F2_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Kir2.1',
    tags=['DANs', 'PAM', '58E02', 'Kir2.1'],
)
# sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261102\\261102_F2_C1\\LEDFlashTriggerPiezoControl_261102_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 630.0, 350.0, -280.0, 60.0): 400, ('lo', 630.0, 450.0, -180.0, 60.0): 400})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 91; no_as_mv: 170; as_off: 529; success: 107; hard_success: 15
8
0 timeouts
Probe positions dataframe created with 800 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261102_F2_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_nofilt_1_800.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.25 (182627/728282)
hi (1-800)
Saving figure: ./figpanels/261102_F2_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_hi_1_800.png
./figpanels\exports/261102_F2_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_hi_1_800.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.32 (232534/724130)
lo (1-800)
Saving figure: ./figpanels/261102_F2_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_lo_1_800.png
./figpanels\exports/261102_F2_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_lo_1_800.pkl
Addin

### 261102_F3_C1
Purpose "PAM DAN silencing".

In [7]:
# [D3-261102_F3_C1]
id = '261102_F3_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= '58E02>UAS-Kir2.1',
    tags=['DANs', 'PAM', '58E02', 'Kir2.1'],
)
sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261102\\261102_F3_C1\\LEDFlashTriggerPiezoControl_261102_F3_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('lo', 630.0, 450.0, -180.0, 60.0): 588, ('hi', 630.0, 350.0, -280.0, 60.0): 578})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 22; no_as_mv: 175; as_off: 635; success: 59; hard_success: 6
8
297 of 297 timeouts (100.0%) during blue light
Probe positions dataframe created with 1166 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261102_F3_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_nofilt_1_1167.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.03 (45033/1316627)
hi (1-1167)
Saving figure: ./figpanels/261102_F3_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_hi_1_1167.png
./figpanels\exports/261102_F3_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_hi_1_1167.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.12 (150108/1294063)
lo (1-1167)
Saving figure: ./figpanels/261102_F3_C1_+;58E02-GAL4_UAS-Kir21_probe_dist_lo_1_1167.png
./figpanels\exports/261102_F3_C1_+;58E02-GA

,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost,rest_time_fraction,drift_time_fraction,move_time_fraction
241205_F2_C1,None,LEDFlashTriggerPiezoControl_241205_F2_C1_Table...,iav_Kir2_1,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4592.04334,1.037038e+05,0.083984,0.630859,...,epi_only,0.371842,0.024624,3.129737e+06,64.0,8.0,5.745730e+06,0.473787,0.082867,0.443346
241205_F3_C1,None,LEDFlashTriggerPiezoControl_241205_F3_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4601.99376,1.995216e+05,0.003891,0.762646,...,epi_only,0.264464,0.049193,6.841562e+06,20.0,0.0,7.592188e+06,0.304387,0.184354,0.511259
241219_F1_C1,None,LEDFlashTriggerPiezoControl_241219_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",8427.33492,2.468955e+05,0.000000,0.759193,...,epi_only,0.145026,0.023368,6.295168e+06,0.0,0.0,8.077882e+06,0.705697,0.020188,0.274115
241219_F2_C1,None,LEDFlashTriggerPiezoControl_241219_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",5633.13292,1.485294e+05,0.000000,0.703642,...,epi_only,0.212010,0.043292,4.128245e+06,14.0,0.0,6.042135e+06,0.521343,0.093810,0.384847
241220_F1_C1,None,LEDFlashTriggerPiezoControl_241220_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",6633.59082,2.480747e+05,0.000000,0.798077,...,epi_only,0.221039,0.027496,6.949634e+06,11.0,0.0,8.077262e+06,0.634160,0.051451,0.314389
241220_F2_C1,None,LEDFlashTriggerPiezoControl_241220_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",7837.07546,2.913358e+05,0.007212,0.713942,...,epi_only,0.213848,0.023861,8.781938e+06,28.0,0.0,1.333849e+07,0.464890,0.127497,0.407613
250127_F1_C1,None,LEDFlashTriggerPiezoControl_250127_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6800.32020,3.596394e+05,0.027510,0.707015,...,epi_only,0.171928,0.039667,8.867315e+06,42.0,3.0,7.057058e+06,0.420250,0.182454,0.397296
250127_F2_C1,None,LEDFlashTriggerPiezoControl_250127_F2_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",5484.01952,3.742404e+05,0.033670,0.680135,...,epi_only,0.090299,0.055188,1.212134e+07,34.0,2.0,7.109523e+06,0.219265,0.221283,0.559452
250128_F1_C1,None,LEDFlashTriggerPiezoControl_250128_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6574.20504,5.108770e+05,0.098901,0.649725,...,epi_only,0.122484,0.046467,2.343239e+07,54.0,8.0,2.088457e+07,0.238110,0.104299,0.657591
250204_F1_C1,None,LEDFlashTriggerPiezoControl_250204_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",7588.12330,3.860837e+05,0.112965,0.541720,...,epi_only,0.205880,0.090884,1.855656e+07,100.0,15.0,1.836187e+07,0.494042,0.046426,0.459532


# PPL1 - MB504B (https://splitgal4.janelia.org/precomputed/MB%20Paper%202014.html)

In [ ]:
id = '260527_F1_C1'

# T = T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# run_plots(T)

# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'MB504B>UAS-Tetx',
    tags=['DANs','PPL'],
)

In [ ]:
id = '260527_F2_C1'
# THis one is bad, don't include

# T = T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

# T.extract_trial_properties()

# run_plots(T)
# sinq.add_table(table=T)
# sinq.df

In [ ]:
id = '260527_F4_C1'

# T = T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()

# end_of_day_trials = T.df.loc[711:]
# T.exclude_list_of_trials(end_of_day_trials.index,'just not watching')
# T.exclude_trials()


# run_plots(T)
# sinq.add_table(table=T)
# sinq.df

sinq.set_note(
    dayflycell = id,
    text= 'MB504B>UAS-Tetx',
    tags=['DANs','PPL'],
)

## MB504B>UAS-Kir2.1 - Kir2.1 silencing

### 261104_F1_C1
Purpose "PPL1 MB504B >Kir2.1". EMG in proximal femur (also in dataset_5). Pyas failed at 457, restarted. Calibration done this day.

In [8]:
# [D3-261104_F1_C1]
id = '261104_F1_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# Pyas dropped out at 457: check those trials came out as info / excluded, e.g.
# T.exclude_list_of_trials(T.df.loc[457:457].index, 'pyas dropout'); T.exclude_trials()
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= 'MB504B>UAS-Kir2.1',
    tags=['DANs', 'PPL', 'MB504B', 'Kir2.1'],
)
sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261104\\261104_F1_C1\\LEDFlashTriggerPiezoControl_261104_F1_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 630.0, 350.0, -280.0, 60.0): 400, ('lo', 630.0, 450.0, -180.0, 60.0): 307})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 5; no_as_mv: 117; as_off: 582; success: 58; hard_success: 1
8
0 timeouts
Probe positions dataframe created with 707 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261104_F1_C1_+;MB504B_UAS-Kir21_probe_dist_nofilt_1_707.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.05 (36332/694963)
hi (1-707)
Saving figure: ./figpanels/261104_F1_C1_+;MB504B_UAS-Kir21_probe_dist_hi_1_707.png
./figpanels\exports/261104_F1_C1_+;MB504B_UAS-Kir21_probe_dist_hi_1_707.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.15 (87953/586814)
lo (1-707)
Saving figure: ./figpanels/261104_F1_C1_+;MB504B_UAS-Kir21_probe_dist_lo_1_707.png
./figpanels\exports/261104_F1_C1_+;MB504B_UAS-Kir21_probe_dist_lo_1_707.pkl
Adding Table for 261104_F1_C1


,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost,rest_time_fraction,drift_time_fraction,move_time_fraction
241205_F2_C1,None,LEDFlashTriggerPiezoControl_241205_F2_C1_Table...,iav_Kir2_1,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4592.04334,1.037038e+05,0.083984,0.630859,...,epi_only,0.371842,0.024624,3.129737e+06,64.0,8.0,5.745730e+06,0.473787,0.082867,0.443346
241205_F3_C1,None,LEDFlashTriggerPiezoControl_241205_F3_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4601.99376,1.995216e+05,0.003891,0.762646,...,epi_only,0.264464,0.049193,6.841562e+06,20.0,0.0,7.592188e+06,0.304387,0.184354,0.511259
241219_F1_C1,None,LEDFlashTriggerPiezoControl_241219_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",8427.33492,2.468955e+05,0.000000,0.759193,...,epi_only,0.145026,0.023368,6.295168e+06,0.0,0.0,8.077882e+06,0.705697,0.020188,0.274115
241219_F2_C1,None,LEDFlashTriggerPiezoControl_241219_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",5633.13292,1.485294e+05,0.000000,0.703642,...,epi_only,0.212010,0.043292,4.128245e+06,14.0,0.0,6.042135e+06,0.521343,0.093810,0.384847
241220_F1_C1,None,LEDFlashTriggerPiezoControl_241220_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",6633.59082,2.480747e+05,0.000000,0.798077,...,epi_only,0.221039,0.027496,6.949634e+06,11.0,0.0,8.077262e+06,0.634160,0.051451,0.314389
241220_F2_C1,None,LEDFlashTriggerPiezoControl_241220_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",7837.07546,2.913358e+05,0.007212,0.713942,...,epi_only,0.213848,0.023861,8.781938e+06,28.0,0.0,1.333849e+07,0.464890,0.127497,0.407613
250127_F1_C1,None,LEDFlashTriggerPiezoControl_250127_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6800.32020,3.596394e+05,0.027510,0.707015,...,epi_only,0.171928,0.039667,8.867315e+06,42.0,3.0,7.057058e+06,0.420250,0.182454,0.397296
250127_F2_C1,None,LEDFlashTriggerPiezoControl_250127_F2_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",5484.01952,3.742404e+05,0.033670,0.680135,...,epi_only,0.090299,0.055188,1.212134e+07,34.0,2.0,7.109523e+06,0.219265,0.221283,0.559452
250128_F1_C1,None,LEDFlashTriggerPiezoControl_250128_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6574.20504,5.108770e+05,0.098901,0.649725,...,epi_only,0.122484,0.046467,2.343239e+07,54.0,8.0,2.088457e+07,0.238110,0.104299,0.657591
250204_F1_C1,None,LEDFlashTriggerPiezoControl_250204_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",7588.12330,3.860837e+05,0.112965,0.541720,...,epi_only,0.205880,0.090884,1.855656e+07,100.0,15.0,1.836187e+07,0.494042,0.046426,0.459532


### 261104_F2_C1
Purpose "More PPL silencing expts."

In [9]:
# [D3-261104_F2_C1]
id = '261104_F2_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

# if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
#     sinq.add_table(table=T)
# sinq.set_note(
#     dayflycell = id,
#     text= 'MB504B>UAS-Kir2.1',
#     tags=['DANs', 'PPL', 'MB504B', 'Kir2.1'],
# )
# sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261104\\261104_F2_C1\\LEDFlashTriggerPiezoControl_261104_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 630.0, 350.0, -280.0, 60.0): 405, ('lo', 630.0, 450.0, -180.0, 60.0): 399})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 145; no_as_mv: 100; as_off: 554; success: 49; hard_success: 8
8
0 timeouts
Probe positions dataframe created with 804 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261104_F2_C1_+;MB504B_UAS-Kir21_probe_dist_nofilt_1_806.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.07 (54686/747085)
hi (1-806)
Saving figure: ./figpanels/261104_F2_C1_+;MB504B_UAS-Kir21_probe_dist_hi_1_806.png
./figpanels\exports/261104_F2_C1_+;MB504B_UAS-Kir21_probe_dist_hi_1_806.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.43 (318262/748182)
lo (1-806)
Saving figure: ./figpanels/261104_F2_C1_+;MB504B_UAS-Kir21_probe_dist_lo_1_806.png
./figpanels\exports/261104_F2_C1_+;MB504B_UAS-Kir21_probe_dist_lo_1_806.pkl


# MB010B (KCs) > UAS-Kir2.1
Kenyon cell silencing. 261030_F1/F2 were filed as MB504B; the 261030_F3 notes say every fly that day had the same genotype, and the Acquisition files were corrected to MB010B on 2026-11-08. The 261030_F4 notes say "didn't see anything in the MB010B" - check expression.

### 261030_F1_C1
Purpose "Are KCs required for learning?". Bubbles in the saline today. Recorded from the proximal oblique bristle, unstable, done ~128. Behavior to ~500.

In [10]:
# [D3-261030_F1_C1]
id = '261030_F1_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= 'MB010B>UAS-Kir2.1',
    tags=['KC', 'MB010B', 'Kir2.1'],
)
sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261030\\261030_F1_C1\\LEDFlashTriggerPiezoControl_261030_F1_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 640.0, 360.0, -280.0, 60.0): 302, ('lo', 640.0, 460.0, -180.0, 60.0): 300})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 68; no_as_mv: 108; as_off: 424; success: 60; hard_success: 12
8
0 timeouts
Probe positions dataframe created with 602 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261030_F1_C1_+;MB010B_UAS-Kir21_probe_dist_nofilt_1_602.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.12 (65967/552401)
hi (1-602)
Saving figure: ./figpanels/261030_F1_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_602.png
./figpanels\exports/261030_F1_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_602.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.39 (215383/553196)
lo (1-602)
Saving figure: ./figpanels/261030_F1_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_602.png
./figpanels\exports/261030_F1_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_602.pkl
Adding Table for 261030_F1_

,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost,rest_time_fraction,drift_time_fraction,move_time_fraction
241205_F2_C1,None,LEDFlashTriggerPiezoControl_241205_F2_C1_Table...,iav_Kir2_1,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4592.04334,1.037038e+05,0.083984,0.630859,...,epi_only,0.371842,0.024624,3.129737e+06,64.0,8.0,5.745730e+06,0.473787,0.082867,0.443346
241205_F3_C1,None,LEDFlashTriggerPiezoControl_241205_F3_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4601.99376,1.995216e+05,0.003891,0.762646,...,epi_only,0.264464,0.049193,6.841562e+06,20.0,0.0,7.592188e+06,0.304387,0.184354,0.511259
241219_F1_C1,None,LEDFlashTriggerPiezoControl_241219_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",8427.33492,2.468955e+05,0.000000,0.759193,...,epi_only,0.145026,0.023368,6.295168e+06,0.0,0.0,8.077882e+06,0.705697,0.020188,0.274115
241219_F2_C1,None,LEDFlashTriggerPiezoControl_241219_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",5633.13292,1.485294e+05,0.000000,0.703642,...,epi_only,0.212010,0.043292,4.128245e+06,14.0,0.0,6.042135e+06,0.521343,0.093810,0.384847
241220_F1_C1,None,LEDFlashTriggerPiezoControl_241220_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",6633.59082,2.480747e+05,0.000000,0.798077,...,epi_only,0.221039,0.027496,6.949634e+06,11.0,0.0,8.077262e+06,0.634160,0.051451,0.314389
241220_F2_C1,None,LEDFlashTriggerPiezoControl_241220_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",7837.07546,2.913358e+05,0.007212,0.713942,...,epi_only,0.213848,0.023861,8.781938e+06,28.0,0.0,1.333849e+07,0.464890,0.127497,0.407613
250127_F1_C1,None,LEDFlashTriggerPiezoControl_250127_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6800.32020,3.596394e+05,0.027510,0.707015,...,epi_only,0.171928,0.039667,8.867315e+06,42.0,3.0,7.057058e+06,0.420250,0.182454,0.397296
250127_F2_C1,None,LEDFlashTriggerPiezoControl_250127_F2_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",5484.01952,3.742404e+05,0.033670,0.680135,...,epi_only,0.090299,0.055188,1.212134e+07,34.0,2.0,7.109523e+06,0.219265,0.221283,0.559452
250128_F1_C1,None,LEDFlashTriggerPiezoControl_250128_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6574.20504,5.108770e+05,0.098901,0.649725,...,epi_only,0.122484,0.046467,2.343239e+07,54.0,8.0,2.088457e+07,0.238110,0.104299,0.657591
250204_F1_C1,None,LEDFlashTriggerPiezoControl_250204_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",7588.12330,3.860837e+05,0.112965,0.541720,...,epi_only,0.205880,0.090884,1.855656e+07,100.0,15.0,1.836187e+07,0.494042,0.046426,0.459532


### 261030_F2_C1

In [11]:
# [D3-261030_F2_C1]
id = '261030_F2_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= 'MB010B>UAS-Kir2.1',
    tags=['KC', 'MB010B', 'Kir2.1'],
)
sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261030\\261030_F2_C1\\LEDFlashTriggerPiezoControl_261030_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 620.0, 340.0, -280.0, 60.0): 300, ('lo', 620.0, 440.0, -180.0, 60.0): 300})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 5; no_as_mv: 88; as_off: 418; success: 32; hard_success: 1
8
51 of 51 timeouts (100.0%) during blue light
Probe positions dataframe created with 600 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261030_F2_C1_+;MB010B_UAS-Kir21_probe_dist_nofilt_1_600.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.05 (25805/571710)
hi (1-600)
Saving figure: ./figpanels/261030_F2_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_600.png
./figpanels\exports/261030_F2_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_600.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.13 (74416/572098)
lo (1-600)
Saving figure: ./figpanels/261030_F2_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_600.png
./figpanels\exports/261030_F2_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_600.p

,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost,rest_time_fraction,drift_time_fraction,move_time_fraction
241205_F2_C1,None,LEDFlashTriggerPiezoControl_241205_F2_C1_Table...,iav_Kir2_1,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4592.04334,1.037038e+05,0.083984,0.630859,...,epi_only,0.371842,0.024624,3.129737e+06,64.0,8.0,5.745730e+06,0.473787,0.082867,0.443346
241205_F3_C1,None,LEDFlashTriggerPiezoControl_241205_F3_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4601.99376,1.995216e+05,0.003891,0.762646,...,epi_only,0.264464,0.049193,6.841562e+06,20.0,0.0,7.592188e+06,0.304387,0.184354,0.511259
241219_F1_C1,None,LEDFlashTriggerPiezoControl_241219_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",8427.33492,2.468955e+05,0.000000,0.759193,...,epi_only,0.145026,0.023368,6.295168e+06,0.0,0.0,8.077882e+06,0.705697,0.020188,0.274115
241219_F2_C1,None,LEDFlashTriggerPiezoControl_241219_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",5633.13292,1.485294e+05,0.000000,0.703642,...,epi_only,0.212010,0.043292,4.128245e+06,14.0,0.0,6.042135e+06,0.521343,0.093810,0.384847
241220_F1_C1,None,LEDFlashTriggerPiezoControl_241220_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",6633.59082,2.480747e+05,0.000000,0.798077,...,epi_only,0.221039,0.027496,6.949634e+06,11.0,0.0,8.077262e+06,0.634160,0.051451,0.314389
241220_F2_C1,None,LEDFlashTriggerPiezoControl_241220_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",7837.07546,2.913358e+05,0.007212,0.713942,...,epi_only,0.213848,0.023861,8.781938e+06,28.0,0.0,1.333849e+07,0.464890,0.127497,0.407613
250127_F1_C1,None,LEDFlashTriggerPiezoControl_250127_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6800.32020,3.596394e+05,0.027510,0.707015,...,epi_only,0.171928,0.039667,8.867315e+06,42.0,3.0,7.057058e+06,0.420250,0.182454,0.397296
250127_F2_C1,None,LEDFlashTriggerPiezoControl_250127_F2_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",5484.01952,3.742404e+05,0.033670,0.680135,...,epi_only,0.090299,0.055188,1.212134e+07,34.0,2.0,7.109523e+06,0.219265,0.221283,0.559452
250128_F1_C1,None,LEDFlashTriggerPiezoControl_250128_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6574.20504,5.108770e+05,0.098901,0.649725,...,epi_only,0.122484,0.046467,2.343239e+07,54.0,8.0,2.088457e+07,0.238110,0.104299,0.657591
250204_F1_C1,None,LEDFlashTriggerPiezoControl_250204_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",7588.12330,3.860837e+05,0.112965,0.541720,...,epi_only,0.205880,0.090884,1.855656e+07,100.0,15.0,1.836187e+07,0.494042,0.046426,0.459532


### 261030_F3_C1

In [13]:
# [D3-261030_F3_C1]
# NO TABLE YET: build LEDFlashTriggerPiezoControl_261030_F3_C1_Table in MATLAB first, then uncomment.
id = '261030_F3_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

# if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
#     sinq.add_table(table=T)
# sinq.set_note(
#     dayflycell = id,
#     text= 'MB010B>UAS-Kir2.1',
#     tags=['KC', 'MB010B', 'Kir2.1'],
# )
# sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261030\\261030_F3_C1\\LEDFlashTriggerPiezoControl_261030_F3_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 620.0, 340.0, -280.0, 60.0): 324, ('lo', 620.0, 440.0, -180.0, 60.0): 300})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 8; no_as_mv: 102; as_off: 500; success: 47; hard_success: 3
8
0 timeouts
Probe positions dataframe created with 624 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261030_F3_C1_+;MB010B_UAS-Kir21_probe_dist_nofilt_1_624.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.07 (38560/578344)
hi (1-624)
Saving figure: ./figpanels/261030_F3_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_624.png
./figpanels\exports/261030_F3_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_624.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.15 (81747/547343)
lo (1-624)
Saving figure: ./figpanels/261030_F3_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_624.png
./figpanels\exports/261030_F3_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_624.pkl


### 261104_F3_C1
Purpose "KC silencing". EMG in proximal femur on current_1, not great (also in dataset_5).

In [14]:
# [D3-261104_F3_C1]
id = '261104_F3_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= 'MB010B>UAS-Kir2.1',
    tags=['KC', 'MB010B', 'Kir2.1'],
)
sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261104\\261104_F3_C1\\LEDFlashTriggerPiezoControl_261104_F3_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 2 target positions - Counter({('hi', 625.0, 345.0, -280.0, 60.0): 422, ('lo', 625.0, 445.0, -180.0, 60.0): 399})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 11; no_as_mv: 126; as_off: 670; success: 56; hard_success: 2
8
0 timeouts
Probe positions dataframe created with 821 trials
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins

./figpanels\exports/261104_F3_C1_+;MB010B_UAS-Kir21_probe_dist_nofilt_1_822.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-284.] - [-216.]): 0.05 (38460/812798)
hi (1-822)
Saving figure: ./figpanels/261104_F3_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_822.png
./figpanels\exports/261104_F3_C1_+;MB010B_UAS-Kir21_probe_dist_hi_1_822.pkl
pyasState
New filter or probe bins, recomputing distributions
Counting downsampled probe positions in bins
On target ([-184.] - [-116.]): 0.13 (101486/772442)
lo (1-822)
Saving figure: ./figpanels/261104_F3_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_822.png
./figpanels\exports/261104_F3_C1_+;MB010B_UAS-Kir21_probe_dist_lo_1_822.pkl
Adding Table for 261104_F3_C

,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost,rest_time_fraction,drift_time_fraction,move_time_fraction
241205_F2_C1,None,LEDFlashTriggerPiezoControl_241205_F2_C1_Table...,iav_Kir2_1,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4592.04334,1.037038e+05,0.083984,0.630859,...,epi_only,0.371842,0.024624,3.129737e+06,64.0,8.0,5.745730e+06,0.473787,0.082867,0.443346
241205_F3_C1,None,LEDFlashTriggerPiezoControl_241205_F3_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",4601.99376,1.995216e+05,0.003891,0.762646,...,epi_only,0.264464,0.049193,6.841562e+06,20.0,0.0,7.592188e+06,0.304387,0.184354,0.511259
241219_F1_C1,None,LEDFlashTriggerPiezoControl_241219_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",8427.33492,2.468955e+05,0.000000,0.759193,...,epi_only,0.145026,0.023368,6.295168e+06,0.0,0.0,8.077882e+06,0.705697,0.020188,0.274115
241219_F2_C1,None,LEDFlashTriggerPiezoControl_241219_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",5633.13292,1.485294e+05,0.000000,0.703642,...,epi_only,0.212010,0.043292,4.128245e+06,14.0,0.0,6.042135e+06,0.521343,0.093810,0.384847
241220_F1_C1,None,LEDFlashTriggerPiezoControl_241220_F1_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",6633.59082,2.480747e+05,0.000000,0.798077,...,epi_only,0.221039,0.027496,6.949634e+06,11.0,0.0,8.077262e+06,0.634160,0.051451,0.314389
241220_F2_C1,None,LEDFlashTriggerPiezoControl_241220_F2_C1_Table...,w;iav-GAL4_UAS-Kir21,"{'text': 'IAV Kir2.1 in [w] background', 'tags...",IAV Kir2.1 in [w] background,"blue, green, iav, w",7837.07546,2.913358e+05,0.007212,0.713942,...,epi_only,0.213848,0.023861,8.781938e+06,28.0,0.0,1.333849e+07,0.464890,0.127497,0.407613
250127_F1_C1,None,LEDFlashTriggerPiezoControl_250127_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6800.32020,3.596394e+05,0.027510,0.707015,...,epi_only,0.171928,0.039667,8.867315e+06,42.0,3.0,7.057058e+06,0.420250,0.182454,0.397296
250127_F2_C1,None,LEDFlashTriggerPiezoControl_250127_F2_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",5484.01952,3.742404e+05,0.033670,0.680135,...,epi_only,0.090299,0.055188,1.212134e+07,34.0,2.0,7.109523e+06,0.219265,0.221283,0.559452
250128_F1_C1,None,LEDFlashTriggerPiezoControl_250128_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",6574.20504,5.108770e+05,0.098901,0.649725,...,epi_only,0.122484,0.046467,2.343239e+07,54.0,8.0,2.088457e+07,0.238110,0.104299,0.657591
250204_F1_C1,None,LEDFlashTriggerPiezoControl_250204_F1_C1_Table...,w;pJFRC7;31H05-GAL4,"{'text': 'w', 'tags': ['blue', 'green', 'w'], ...",w,"blue, green, w",7588.12330,3.860837e+05,0.112965,0.541720,...,epi_only,0.205880,0.090884,1.855656e+07,100.0,15.0,1.836187e+07,0.494042,0.046426,0.459532


# DPM > UAS-Kir2.1

### 261030_F4_C1
Purpose "What happens if you silence the DPM neuron?"

In [15]:
# [D3-261030_F4_C1]
id = '261030_F4_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
run_plots(T)

if id not in sinq.df.index:   # set_note on a missing id would add a half-empty row
    sinq.add_table(table=T)
sinq.set_note(
    dayflycell = id,
    text= 'DPM>UAS-Kir2.1',
    tags=['DPM', 'Kir2.1'],
)
# sinq.display_df(show_tags=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\261030\\261030_F4_C1\\LEDFlashTriggerPiezoControl_261030_F4_C1_Table.parquet")
Getting trials
Excluding trials:
[]
Getting all meta keys
[]
Found 3 target positions - Counter({('hi', 670.0, 390.0, -280.0, 60.0): 549, ('lo', 670.0, 490.0, -180.0, 60.0): 501, ('hi', 640.0, 360.0, -280.0, 60.0): 10})
Writing filtercube_status to each trial
Writing fiberLED to each trial


posx and posy should be finite values


no_as_no_mv: 39; no_as_mv: 182; as_off: 815; success: 92; hard_success: 8
8
9 of 9 timeouts (100.0%) during blue light
excluding: Trial(trial=1, 261030_F4_C1, dT=15.73884, ex=False)
self.excluded: [[1]]
Reason for excluding is: probe values too high
Excluded for Exclusion: probe values too high: Trial(trial=1, 261030_F4_C1, dT=15.73884, ex=True)
excluding: Trial(trial=2, 261030_F4_C1, dT=10.29784, ex=False)
self.excluded: [[1]]
Reason for excluding is: probe values too high
Excluded for Exclusion: probe values too high: Trial(trial=2, 261030_F4_C1, dT=10.29784, ex=True)
excluding: Trial(trial=3, 261030_F4_C1, dT=8.9754, ex=False)
self.excluded: [[1]]
Reason for excluding is: probe values too high
Excluded for Exclusion: probe values too high: Trial(trial=3, 261030_F4_C1, dT=8.9754, ex=True)
excluding: Trial(trial=4, 261030_F4_C1, dT=10.54638, ex=False)
self.excluded: [[1]]
Reason for excluding is: probe values too high
Excluded for Exclusion: probe values too high: Trial(trial=4, 26103

# Effector tags
The PAM and PPL groups mix TeTx and Kir2.1 (and TH vs MB504B drivers for PPL). `add_tags` adds without touching the note text, so the groups can be split by effector.

In [ ]:
# [D3-EFFECTOR-TAGS]
for dfc in ['251114_F1_C2', '260605_F1_C1', '260605_F2_C1', '260605_F3_C1']:
    sinq.add_tags(dfc, ['TeTx', '58E02'])
for dfc in ['260527_F1_C1', '260527_F4_C1']:
    sinq.add_tags(dfc, ['TeTx', 'MB504B'])
for dfc in ['250506_F1_C1', '250506_F2_C1', '250507_F1_C1', '250507_F2_C1', '250509_F2_C1', '250514_F1_C1']:
    sinq.add_tags(dfc, ['Kir2.1'])   # TH-GAL4>Kir2.1, already tagged TH
sinq.display_df(show_tags=True)